# Traffic Demand Prediction — Final Solution

## Root Cause Analysis
- Previous models (v1-v4) scored 86-88 because LightGBM was **overfitting day48 patterns**
  and not generalising to the test distribution.
- **geo_ts_mean** (geohash × 15-min time_slot, from full train) gives **R² = 0.883** on
  day49 validation *with zero model overhead*.
- Adding a complex LGB model on top consistently **hurt** the predictions.

## Why geo_ts_mean works
- Test = day49 hours 2:15–13:45. Day49 in train only has hours 0–2.
- So for all test slots (9–55),  (identical).
- The 15-min slot captures both location baseline AND time-of-day pattern.

## Fallback chain for 11% missing rows
geo_ts (interpolated) → prefix4×slot → prefix3×slot → geo×hour → geo_mean → global_mean


In [ ]:
"""
FINAL CORRECT SOLUTION:
- geo_ts_mean (geohash x time_slot, full train) is the best single predictor (R²=0.883)
- For 11% missing: use interpolated pivot (linear interp across slots per geohash)  
- LGB hurts if too complex. Use VERY minimal LGB only to handle edge cases.
- DO NOT let the model override the geo_ts_mean signal.
"""
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('/home/claude/dataset/dataset/train.csv')
test  = pd.read_csv('/home/claude/dataset/dataset/test.csv')

for df in [train, test]:
    df['hour']        = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute']      = df['timestamp'].str.split(':').str[1].astype(int)
    df['time_slot']   = df['hour'] * 4 + df['minute'] // 15
    df['geo_prefix4'] = df['geohash'].str[:4]
    df['geo_prefix3'] = df['geohash'].str[:3]

global_mean = train['demand'].mean()

# ─── Build interpolated geo x time_slot matrix (from FULL train) ─────────────
pivot = train.groupby(['geohash','time_slot'])['demand'].mean().unstack()
pivot_interp = pivot.interpolate(axis=1, method='linear', limit_direction='both')

# Geo x hour (for fallback, 96.3% coverage)
geo_hour_map = train.groupby(['geohash','hour'])['demand'].mean()
# Prefix4 x timeslot
gp4_ts_map   = train.groupby(['geo_prefix4','time_slot'])['demand'].mean()
# Prefix3 x timeslot
gp3_ts_map   = train.groupby(['geo_prefix3','time_slot'])['demand'].mean()
# Prefix4 x hour
gp4_hour_map = train.groupby(['geo_prefix4','hour'])['demand'].mean()
# Geo overall mean
geo_mean_map = train.groupby('geohash')['demand'].mean()
# Prefix means
gp4_mean_map = train.groupby('geo_prefix4')['demand'].mean()
gp3_mean_map = train.groupby('geo_prefix3')['demand'].mean()
# Global slot/hour patterns
slot_mean_map = train.groupby('time_slot')['demand'].mean()
hour_mean_map = train.groupby('hour')['demand'].mean()

def get_geo_ts_interp(geo, ts):
    """Get interpolated geo x time_slot demand"""
    if geo in pivot_interp.index and ts in pivot_interp.columns:
        v = pivot_interp.loc[geo, ts]
        if not np.isnan(v):
            return v
    return np.nan

def build_pred_features(df):
    df = df.copy()

    # Primary: geo x ts (exact, then interpolated)
    df['geo_ts'] = df.apply(lambda r: get_geo_ts_interp(r['geohash'], r['time_slot']), axis=1)

    # Secondary: geo x hour
    df['geo_hour'] = df.apply(lambda r: geo_hour_map.get((r['geohash'], r['hour']), np.nan), axis=1)

    # Fallback chain
    df['gp4_ts']   = df.apply(lambda r: gp4_ts_map.get((r['geo_prefix4'], r['time_slot']), np.nan), axis=1)
    df['gp3_ts']   = df.apply(lambda r: gp3_ts_map.get((r['geo_prefix3'], r['time_slot']), np.nan), axis=1)
    df['gp4_hour'] = df.apply(lambda r: gp4_hour_map.get((r['geo_prefix4'], r['hour']), np.nan), axis=1)
    df['geo_mean'] = df['geohash'].map(geo_mean_map).fillna(global_mean)
    df['gp4_mean'] = df['geo_prefix4'].map(gp4_mean_map).fillna(global_mean)
    df['gp3_mean'] = df['geo_prefix3'].map(gp3_mean_map).fillna(global_mean)
    df['slot_mean'] = df['time_slot'].map(slot_mean_map).fillna(global_mean)
    df['hour_mean'] = df['hour'].map(hour_mean_map).fillna(global_mean)

    # Neighbor slots (from interpolated pivot)
    for d in [-1, 1, -2, 2, -4, 4]:
        df[f'geo_ts_{d:+d}'] = df.apply(
            lambda r: get_geo_ts_interp(r['geohash'], (r['time_slot'] + d) % 96), axis=1)

    # Neighbor hours
    for d in [-1, 1]:
        df[f'geo_hr_{d:+d}'] = df.apply(
            lambda r: geo_hour_map.get((r['geohash'], (r['hour'] + d) % 24), np.nan), axis=1)

    # Fill nulls
    fallback = (df['gp4_ts'].fillna(df['gp3_ts']).fillna(df['geo_hour'])
                .fillna(df['gp4_hour']).fillna(df['geo_mean'])
                .fillna(df['slot_mean']).fillna(global_mean))
    df['geo_ts'] = df['geo_ts'].fillna(fallback)
    df['geo_hour'] = df['geo_hour'].fillna(df['gp4_hour']).fillna(df['geo_mean']).fillna(df['hour_mean']).fillna(global_mean)
    for d in [-1,1,-2,2,-4,4]:
        df[f'geo_ts_{d:+d}'] = df[f'geo_ts_{d:+d}'].fillna(df['geo_ts'])
    for d in [-1,1]:
        df[f'geo_hr_{d:+d}'] = df[f'geo_hr_{d:+d}'].fillna(df['geo_hour'])
    for c in ['gp4_ts','gp3_ts','gp4_hour']:
        df[c] = df[c].fillna(df['geo_mean'])

    # Derived interaction features
    df['ts_vs_hour']  = df['geo_ts'] / (df['geo_hour'] + 1e-8)
    df['ts_trend']    = df['geo_ts_+1'] - df['geo_ts_-1']
    df['hr_trend']    = df['geo_hr_+1'] - df['geo_hr_-1']
    df['geo_vs_slot'] = df['geo_mean'] / (df['slot_mean'] + 1e-8)

    # Cyclic time
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 96)

    # Road/weather
    df['RoadType_enc']      = df['RoadType'].map({'Residential':0,'Street':1,'Highway':2}).fillna(-1).astype(int)
    df['LargeVehicles_enc'] = (df['LargeVehicles']=='Allowed').astype(int)
    df['Landmarks_enc']     = (df['Landmarks']=='Yes').astype(int)
    df['Weather_enc']       = df['Weather'].map({'Sunny':0,'Cloudy':1,'Rainy':2,'Foggy':3,'Snowy':4}).fillna(-1).astype(int)
    tmp = df.groupby('Weather')['Temperature'].transform('median')
    df['Temperature'] = df['Temperature'].fillna(tmp).fillna(df['Temperature'].median())
    df['capacity_proxy'] = df['NumberofLanes'] * (df['RoadType_enc'] + 1)

    return df

print("Building features...")
train_f = build_pred_features(train)
test_f  = build_pred_features(test)

# Validate on day49
X_d49 = train_f[train_f['day'] == 49]
print(f"\nDirect geo_ts R² on day49: {r2_score(X_d49['demand'], X_d49['geo_ts']):.5f}")

FEATURES = [
    'geo_ts', 'geo_hour',
    'geo_ts_-1', 'geo_ts_+1', 'geo_ts_-2', 'geo_ts_+2', 'geo_ts_-4', 'geo_ts_+4',
    'geo_hr_-1', 'geo_hr_+1',
    'gp4_ts', 'gp3_ts', 'gp4_hour',
    'geo_mean', 'gp4_mean', 'gp3_mean',
    'slot_mean', 'hour_mean',
    'ts_vs_hour', 'ts_trend', 'hr_trend', 'geo_vs_slot',
    'hour', 'minute', 'time_slot',
    'hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
    'NumberofLanes', 'RoadType_enc', 'LargeVehicles_enc',
    'Landmarks_enc', 'Weather_enc', 'Temperature', 'capacity_proxy',
]

print(f"Features: {len(FEATURES)}, NaN: {train_f[FEATURES].isnull().sum().sum()}")

# ─── Very conservative LGB model ──────────────────────────────────────────────
X_d48 = train_f[train_f['day'] == 48]

params = dict(
    n_estimators=5000, learning_rate=0.005,   # very slow learning rate
    num_leaves=15, max_depth=4,                # very shallow trees
    feature_fraction=0.6, bagging_fraction=0.6, bagging_freq=5,
    min_child_samples=100,                     # high min samples
    reg_alpha=1.0, reg_lambda=2.0,             # strong regularization
    random_state=42, verbose=-1
)

print("\n=== Conservative LGB (Day48 → Day49) ===")
vm = lgb.LGBMRegressor(**params)
vm.fit(X_d48[FEATURES], X_d48['demand'],
       eval_set=[(X_d49[FEATURES], X_d49['demand'])],
       callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(500)])
vm_pred = vm.predict(X_d49[FEATURES])
r2_lgb = r2_score(X_d49['demand'], vm_pred)
print(f"LGB R² on day49: {r2_lgb:.5f}, best_iter={vm.best_iteration_}")

# ─── Choose best approach ─────────────────────────────────────────────────────
r2_direct = r2_score(X_d49['demand'], X_d49['geo_ts'])
print(f"\nDirect geo_ts: {r2_direct:.5f}")
print(f"LGB:          {r2_lgb:.5f}")
print(f"Blend 50/50:  {r2_score(X_d49['demand'], 0.5*X_d49['geo_ts']+0.5*vm_pred):.5f}")
print(f"Blend 70/30:  {r2_score(X_d49['demand'], 0.7*X_d49['geo_ts']+0.3*vm_pred):.5f}")
print(f"Blend 30/70:  {r2_score(X_d49['demand'], 0.3*X_d49['geo_ts']+0.7*vm_pred):.5f}")

# ─── Final model on all train ─────────────────────────────────────────────────
best_n = max(vm.best_iteration_, 50)
print(f"\n=== Final model (n={best_n}) ===")
fm = lgb.LGBMRegressor(**{**params, 'n_estimators': best_n})
fm.fit(train_f[FEATURES], train_f['demand'])

lgb_test = fm.predict(test_f[FEATURES])

# Direct + LGB blend
direct_test = test_f['geo_ts'].values
blend_test  = np.clip(0.5 * direct_test + 0.5 * lgb_test, 0, 1)
direct_clip = np.clip(direct_test, 0, 1)

fi = pd.Series(fm.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nTop 10 features:")
print(fi.head(10).to_string())

for name, p in [('direct_geo_ts', direct_clip), ('lgb_only', np.clip(lgb_test,0,1)),
                ('blend_5050', blend_test)]:
    s = pd.DataFrame({'Index': test['Index'], 'demand': p})
    s.to_csv(f'/mnt/user-data/outputs/sub_{name}.csv', index=False)
    print(f"\nsub_{name}: mean={p.mean():.4f}, std={p.std():.4f}")
